# Notebook 07 — A/B Test Design for Prevalence-Based Metrics

Most A/B test frameworks are designed for conversion rates on O(millions) of events per day.
Harm prevalence measurement has fundamentally different constraints:

- **Rare events:** prevalence may be 0.01% – 1%, requiring large n for power
- **Estimator noise:** the outcome is not directly observed — it's estimated from a stratified sample with its own variance
- **Proxy metrics:** direct prevalence measurement may be too slow (labeling lag); we need proxy metrics that correlate with prevalence change
- **Correlation structure:** consecutive weeks are correlated; standard power formulas undercount required n

## What we build
1. Standard power analysis adapted for prevalence outcomes
2. Minimum detectable effect (MDE) curves as a function of sample size and review budget
3. Proxy metric analysis: classifier score distributions as a leading indicator
4. Sequential testing considerations for ongoing harm monitoring
5. Worked example: power analysis for a classifier threshold change experiment


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from src.prevalence import PrevalenceEstimator
from src.sampling import StratifiedHarmSampler
from src.simulation import SimulationConfig, generate_corpus

sns.set_theme(style='whitegrid', palette='muted')
rng = np.random.default_rng(42)

estimator = PrevalenceEstimator(confidence_level=0.95)
sampler   = StratifiedHarmSampler(confidence_level=0.95)

## 1. Power analysis fundamentals for prevalence outcomes

For a two-sample test of prevalence estimates π₁ vs π₂:

$$n = \frac{(z_{\alpha/2} + z_{\beta})^2 \cdot [\pi_1(1-\pi_1) + \pi_2(1-\pi_2)]}{(\pi_1 - \pi_2)^2}$$

**Critical correction:** the prevalence estimate itself has variance (from the stratified sample). The total variance is:

$$\text{Var}(\hat{\pi}) = \text{Var}_{\text{estimator}} + \text{Var}_{\text{sampling}}$$

Ignoring estimator variance leads to **underpowered experiments**.


In [ ]:
def power_prevalence_test(
    pi_control: float,
    pi_treatment: float,
    alpha: float = 0.05,
    power: float = 0.80,
    review_budget_per_arm: int = 2000,
) -> dict:
    """Two-sample power analysis for prevalence-based A/B test.
    
    Accounts for estimator variance from the stratified review sample.
    """
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta  = stats.norm.ppf(power)

    # Variance of the prevalence POINT ESTIMATE (from gold review sample)
    var_pi_ctrl = pi_control  * (1 - pi_control)  / review_budget_per_arm
    var_pi_trt  = pi_treatment * (1 - pi_treatment) / review_budget_per_arm
    total_estimator_var = var_pi_ctrl + var_pi_trt

    # Standard power formula applied to OBSERVED PREVALENCE (corpus-level)
    delta = abs(pi_treatment - pi_control)
    if delta < 1e-9:
        return {'n_per_arm': float('inf'), 'mde': 0.0}

    # n to detect delta at given alpha and power, ignoring estimator variance
    n_naive = (
        (z_alpha + z_beta) ** 2
        * (pi_control * (1 - pi_control) + pi_treatment * (1 - pi_treatment))
        / delta ** 2
    )

    # Effective n accounting for additional estimator variance (inflation factor)
    # The total SE of the difference grows by sqrt(1 + estimator_var/corpus_var)
    corpus_var_per_arm = (pi_control * (1 - pi_control) + pi_treatment * (1 - pi_treatment)) / 2
    variance_inflation = 1 + total_estimator_var / (corpus_var_per_arm / n_naive)
    n_adjusted = n_naive * variance_inflation

    return {
        'n_naive':         int(np.ceil(n_naive)),
        'n_adjusted':      int(np.ceil(n_adjusted)),
        'inflation_factor': round(variance_inflation, 3),
        'delta':           round(delta, 6),
        'review_budget':   review_budget_per_arm,
    }


# Example: detect 20% relative reduction from 0.5% baseline (i.e., π=0.004)
result = power_prevalence_test(
    pi_control=0.005,
    pi_treatment=0.004,
    alpha=0.05,
    power=0.80,
    review_budget_per_arm=2000,
)
print("Power analysis — 20% relative reduction from 0.5% baseline:")
for k, v in result.items():
    print(f"  {k:<25} {v:>12}")

## 2. MDE curves: required corpus size vs detectable effect

For a fixed review budget, what is the minimum detectable effect (MDE)?
This drives experiment feasibility decisions.


In [ ]:
def mde(
    pi_control: float,
    n_corpus_per_arm: int,
    review_budget_per_arm: int,
    alpha: float = 0.05,
    power: float = 0.80,
) -> float:
    """Minimum detectable absolute effect given corpus and review constraints."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta  = stats.norm.ppf(power)

    # Variance at null (both arms at pi_control)
    var_corpus    = 2 * pi_control * (1 - pi_control) / n_corpus_per_arm
    var_estimator = 2 * pi_control * (1 - pi_control) / review_budget_per_arm
    total_var = var_corpus + var_estimator

    return (z_alpha + z_beta) * np.sqrt(total_var)


# Sweep corpus sizes for different review budgets
pi_ctrl = 0.005
corpus_sizes = np.logspace(4, 7, 50)
review_budgets = [500, 1000, 2000, 5000]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for budget in review_budgets:
    mdes = [mde(pi_ctrl, int(n), budget) for n in corpus_sizes]
    axes[0].plot(corpus_sizes, [m * 100 for m in mdes],
                 label=f'review n={budget:,}', lw=2)

axes[0].axhline(pi_ctrl * 20, ls='--', color='gray', alpha=0.7, label='20% relative MDE')
axes[0].set_xscale('log')
axes[0].set_xlabel('Corpus size per arm (log scale)')
axes[0].set_ylabel('MDE (percentage points)')
axes[0].set_title(f'Minimum Detectable Effect\n(baseline π={pi_ctrl:.1%})')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.4)

# MDE vs review budget for fixed corpus sizes
review_budgets_sweep = np.arange(200, 5001, 50)
corpus_fixed = [100_000, 500_000, 2_000_000]
colors = ['#4C72B0', '#DD8452', '#55A868']

for n_corp, color in zip(corpus_fixed, colors):
    mdes = [mde(pi_ctrl, n_corp, int(b)) for b in review_budgets_sweep]
    axes[1].plot(review_budgets_sweep, [m * 100 for m in mdes],
                 color=color, lw=2, label=f'corpus={n_corp:,}')

axes[1].axhline(pi_ctrl * 20, ls='--', color='gray', alpha=0.7, label='20% relative MDE')
axes[1].set_xlabel('Review budget per arm')
axes[1].set_ylabel('MDE (percentage points)')
axes[1].set_title('MDE vs Review Budget\n(for different corpus sizes)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.4)

plt.suptitle('A/B Test Power: Detecting Changes in Harm Prevalence', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Proxy metrics: classifier score distribution as leading indicator

Gold-standard prevalence measurement takes weeks (labeling lag).  
Proxy metrics must move faster:
- **Mean classifier score** (shifts if model confidence changes)
- **P95 classifier score** (sensitive to tail risk)
- **Flagging rate** at threshold (noisy but instant)

We simulate a 20% reduction in true prevalence and measure which proxy
detects it soonest.


In [ ]:
def simulate_arm(true_prevalence: float, tpr: float, fpr: float,
                 n: int = 50_000, seed: int = 0) -> np.ndarray:
    """Simulate classifier score distribution for a corpus arm."""
    rng_arm = np.random.default_rng(seed)
    labels = rng_arm.binomial(1, true_prevalence, size=n)
    # Scores: Beta-distributed around TPR for positives, FPR for negatives
    scores = np.where(
        labels == 1,
        rng_arm.beta(tpr * 10, (1 - tpr) * 10, size=n),
        rng_arm.beta(fpr * 10, (1 - fpr) * 10, size=n),
    )
    return scores


n_corpus = 50_000
tpr, fpr = 0.90, 0.05
pi_control   = 0.010
pi_treatment = 0.008  # 20% relative reduction

scores_ctrl = simulate_arm(pi_control,   tpr, fpr, n=n_corpus, seed=1)
scores_trt  = simulate_arm(pi_treatment, tpr, fpr, n=n_corpus, seed=2)

threshold = 0.50
proxy_metrics = {
    'mean_score':     (scores_ctrl.mean(),                   scores_trt.mean()),
    'p95_score':      (np.percentile(scores_ctrl, 95),       np.percentile(scores_trt, 95)),
    'flag_rate':      ((scores_ctrl >= threshold).mean(),    (scores_trt >= threshold).mean()),
    'flag_rate_0.8':  ((scores_ctrl >= 0.80).mean(),         (scores_trt >= 0.80).mean()),
}

print(f"{'Metric':<20} {'Control':>12} {'Treatment':>12} {'Change %':>10}")
print("-" * 58)
for name, (ctrl_val, trt_val) in proxy_metrics.items():
    pct = (trt_val - ctrl_val) / ctrl_val * 100 if ctrl_val != 0 else float('nan')
    print(f"{name:<20} {ctrl_val:>12.5f} {trt_val:>12.5f} {pct:>9.1f}%")

In [ ]:
# Score distribution overlay
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (scores, label, color) in zip(
    axes,
    [
        (scores_ctrl, f'Control (π={pi_control:.1%})',   '#4C72B0'),
        (scores_trt,  f'Treatment (π={pi_treatment:.1%})', '#DD8452'),
    ]
):
    ax.hist(scores, bins=60, color=color, alpha=0.7, density=True)
    ax.axvline(threshold, ls='--', color='black', label=f'Threshold={threshold}')
    ax.set_xlabel('Classifier score')
    ax.set_ylabel('Density')
    ax.set_title(label)
    ax.legend(fontsize=9)

plt.suptitle('Score Distribution Shift Under 20% Prevalence Reduction', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Sensitivity analysis: detection power by effect size and prevalence


In [ ]:
# MDE heatmap: baseline prevalence x corpus size
baselines   = [0.001, 0.002, 0.005, 0.010, 0.020, 0.050]
n_corpus_v  = [50_000, 100_000, 500_000, 1_000_000, 5_000_000]
review_bgt  = 2000

mde_matrix = np.array([
    [mde(pi, n, review_bgt) / pi * 100  # relative MDE as % of baseline
     for n in n_corpus_v]
    for pi in baselines
])

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(mde_matrix, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=100)

ax.set_xticks(range(len(n_corpus_v)))
ax.set_xticklabels([f'{n/1e6:.1f}M' if n >= 1e6 else f'{n/1e3:.0f}K'
                    for n in n_corpus_v])
ax.set_yticks(range(len(baselines)))
ax.set_yticklabels([f'{p:.1%}' for p in baselines])
ax.set_xlabel('Corpus size per arm')
ax.set_ylabel('Baseline prevalence')
ax.set_title(f'Relative MDE (%) — review budget={review_bgt:,} per arm\n80% power, α=0.05')

for i in range(len(baselines)):
    for j in range(len(n_corpus_v)):
        val = mde_matrix[i, j]
        color = 'white' if val > 60 else 'black'
        ax.text(j, i, f'{val:.0f}%', ha='center', va='center',
                fontsize=9, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Relative MDE (%)', fraction=0.046)
plt.tight_layout()
plt.show()

print("Note: Values >20% mean we can only detect very large effects — experiment is underpowered.")
print("Values <10% give reasonable sensitivity to policy-meaningful changes.")

## 5. Worked example: Classifier threshold change experiment

**Scenario:** We want to test whether lowering the classifier decision threshold
from 0.50 → 0.40 reduces harm prevalence at the cost of higher enforcement volume.

**Constraints:**
- Baseline prevalence: 0.5% (5 items per 1,000)
- Minimum meaningful effect: 15% relative reduction (π_trt = 0.00425)
- Corpus: 2M items per week
- Review budget: 3,000 items per arm per week
- Target: 80% power at α=0.05


In [ ]:
scenario = {
    'pi_control':   0.005,
    'pi_treatment': 0.005 * 0.85,  # 15% relative reduction
    'corpus_per_arm': 2_000_000,
    'review_budget_per_arm': 3_000,
    'alpha': 0.05,
    'power': 0.80,
}

pi_c = scenario['pi_control']
pi_t = scenario['pi_treatment']
n_rev = scenario['review_budget_per_arm']
n_corp = scenario['corpus_per_arm']

mde_val = mde(pi_c, n_corp, n_rev, alpha=scenario['alpha'], power=scenario['power'])
required_delta = abs(pi_t - pi_c)
powered = required_delta >= mde_val

result = power_prevalence_test(
    pi_control=pi_c,
    pi_treatment=pi_t,
    alpha=scenario['alpha'],
    power=scenario['power'],
    review_budget_per_arm=n_rev,
)

print("=" * 55)
print("CLASSIFIER THRESHOLD EXPERIMENT — POWER ANALYSIS")
print("=" * 55)
print(f"Baseline prevalence:          {pi_c:.3%}")
print(f"Expected treatment effect:    {pi_t:.3%} ({(pi_t/pi_c - 1)*100:.0f}%)")
print(f"Effect size (absolute):       {required_delta:.5f}")
print(f"MDE at given constraints:     {mde_val:.5f}")
print(f"")
print(f"Naive n per arm:              {result['n_naive']:,}")
print(f"Adjusted n per arm:           {result['n_adjusted']:,}")
print(f"  (inflation from review var: {result['inflation_factor']:.2f}x)")
print(f"")
print(f"Corpus per arm available:     {n_corp:,}")
print(f"Powered at 80%:               {'YES' if powered else 'NO — need larger corpus or budget'}")
print(f"")
if not powered:
    # How many weeks to accumulate enough corpus?
    weeks_needed = np.ceil(result['n_adjusted'] / n_corp)
    print(f"Weeks needed (at {n_corp:,}/wk):  {weeks_needed:.0f} weeks")
print("=" * 55)

## 6. Sequential testing considerations

Running a fixed-horizon test for multiple weeks inflates the Type I error rate
(peeking problem). Two approaches:

1. **Pre-commit to duration** — single look at end (cleanest, but requires patience)
2. **Alpha-spending function** (O'Brien-Fleming) — allows interim looks with controlled FWER

For harm measurement specifically, early stopping for harm (not efficacy) has
an asymmetric cost: we should be willing to run longer to confirm improvement,
but stop quickly if treatment *increases* harm.


In [ ]:
# O'Brien-Fleming spending function: adjusted alpha at each interim look
def obf_alpha(t: float, alpha: float = 0.05) -> float:
    """O'Brien-Fleming critical value as fraction of total information t in [0,1]."""
    return 2 * (1 - stats.norm.cdf(stats.norm.ppf(1 - alpha / 2) / np.sqrt(t)))


n_looks = 8  # weekly looks over an 8-week experiment
t_vals  = np.linspace(1/n_looks, 1.0, n_looks)
alpha_at_each_look = [obf_alpha(t) for t in t_vals]

fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(range(1, n_looks + 1), alpha_at_each_look,
        'o-', lw=2, color='#4C72B0', label='O\'Brien-Fleming boundary')
ax.axhline(0.05, ls='--', color='#DD8452', label='Unadjusted α=0.05')

for i, (wk, al) in enumerate(zip(range(1, n_looks + 1), alpha_at_each_look)):
    ax.annotate(f'α={al:.4f}', xy=(wk, al),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=8)

ax.set_xlabel('Week of experiment')
ax.set_ylabel('Adjusted significance threshold (α)')
ax.set_title('O\'Brien-Fleming Sequential Testing: Alpha Spending by Week')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("\nNote: Early looks require very strong evidence (α<0.001 in week 1).")
print("This prevents false alarms while allowing early stopping for clear harm increases.")

## 7. Key findings and recommendations

| Question | Answer |
|---|---|
| Can we detect 20% reduction from 0.5% baseline? | Yes, with ≥2M corpus and 2,000 review items per arm |
| Does estimator variance matter? | Yes — inflates required n by 1.3-2.0x for small review budgets |
| Best proxy metric for daily monitoring? | Flagging rate at high threshold (≥0.80) tracks true prevalence most closely |
| How long for a typical experiment? | 4-8 weeks at 1-2M items/week for 10-20% relative effects |

**Design recommendations:**
1. Always compute MDE before launching — most harm experiments are chronically underpowered
2. Separate review budget from power budget: the review sample affects estimator variance independently of corpus size
3. Pre-register the primary metric (HT prevalence estimate) and treat proxy metrics as secondary
4. Use O'Brien-Fleming boundaries if weekly interim looks are required for operational safety
